# Near-miss financial abstention: full pipeline

Runtime → Change runtime type → **T4 GPU**, then run the cells in order.

1. Setup
2. Pilot (separate items, used only to check the prompt)
3. Main experiment
4. Look at the outputs and download them

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup
Upload the code zip (e.g. `code_fixed.zip`, containing the `HW2/` folder) when prompted.

In [ ]:
import glob, os, shutil, zipfile
from google.colab import files

%cd /content
uploaded = files.upload()  # choose the code zip
zip_name = next(n for n in uploaded if n.endswith('.zip'))
shutil.rmtree('/content/near-miss-fin', ignore_errors=True)
zipfile.ZipFile(zip_name).extractall('/content/near-miss-fin')

# work in the folder that holds config.yaml, wherever it sits inside the zip
root = os.path.dirname(sorted(glob.glob('/content/near-miss-fin/**/config.yaml', recursive=True), key=len)[0])
%cd $root
!pip install -q -r requirements.txt
!ls

## 2. Pilot
20 separate items (seed 7). Check that the models are neither at ceiling nor at floor:
- ANS accuracy should be at least 80% (otherwise failures reflect reading, not abstention)
- substitution rates should not be ~0% or ~100% in every condition

In [ ]:
!python src/generate.py --split pilot
!python src/run.py --data data/pilot.jsonl --all
!python src/analyze.py --results results_pilot --data data/pilot.jsonl --out outputs_pilot

In [ ]:
import pandas as pd
p = pd.read_csv('outputs_pilot/summary.csv')
p.pivot_table(index='condition', columns=['model', 'prompt'], values='substitution').round(2)

## 3. Main experiment
120 items × 11 conditions × 2 models × 2 prompts = 5,280 prompts.

In [ ]:
!python src/generate.py --split main
!python src/run.py --data data/items.jsonl --all
!python src/analyze.py
!python src/extra.py   # style table, overlap check (uses the included results_overlap/), error composition

## 4. Outputs

In [ ]:
print(open('outputs/hypotheses.md').read())
print(open('outputs/table_irr.tex').read())
from IPython.display import Image, display
display(Image('outputs/fig_delta.png', width=500))

In [ ]:
s = pd.read_csv('outputs/summary.csv')
s.pivot_table(index='condition', columns=['model', 'prompt'], values=['substitution', 'abstention']).round(3)

In [ ]:
!cat results/*.meta.json
!zip -qr near_miss_outputs.zip data results results_pilot results_overlap outputs outputs_pilot
from google.colab import files
files.download('near_miss_outputs.zip')